<img align="center" width="1024" src="https://github.com/scadriano/machinelearning/blob/main/images/Yolo26_Home.jpg?raw=true">

# **A Revolução da Visão Computacional**

Visão Computacional usando Ultralytics YOLO! O mais aclamado modelo de deteção de objetos em tempo real.

Originalmente é um acrônimo para "You Only Live Once", que pode ser traduzido como "só se vive uma vez". Foi adaptado por Joseph Redmon para "You Only Look Once", que pode ser traduzido como "Você Só Olha Uma Vez", sendo um algoritmo de detecção de objetos extremamente eficiente e rápido em visão computacional.

Artigo de apresentação do Yolo: https://arxiv.org/abs/1506.02640

Joseph Redmon no TED: https://www.youtube.com/watch?v=Cgxsv1riJhI

**Nesta versão adaptada**, vamos treinar o YOLO com o dataset **Argoverse** (cenas de trânsito urbano: carros, pedestres, ciclistas, ônibus, sinais de trânsito etc.) e depois usar o modelo treinado para analisar, frame a frame, um vídeo enviado por você.

**Não há mágica. Há matemática!** 🧙

---

**#boracomecar -> Antes de tudo!**

Acesse Menu: Ambiente de execução > Alterar o tipo de ambiente de execução > defina o Acelerador de hardware como: GPUs.

---


## **Instalar a biblioteca > Ultralytics**

In [ ]:
# Instalar a biblioteca
!pip install -q -U ultralytics

## **Treinar o YOLO com o dataset Argoverse**

O Ultralytics já vem com o arquivo de configuração `Argoverse.yaml`, que baixa automaticamente o dataset (imagens de câmeras veiculares, com classes como `person`, `bicycle`, `car`, `motorcycle`, `bus`, `truck`, `traffic_light`, `stop_sign`) na primeira execução.

⚠️ **Atenção:** o Argoverse é um dataset grande (dezenas de GB) e o download + treino pode demorar bastante no Colab. Para fins de teste/aula, use poucas épocas (`epochs`) — depois é só aumentar se quiser um modelo mais preciso.


In [ ]:
# Treinar o YOLO26n usando o dataset Argoverse
# epochs baixo (ex: 10-20) para caber no tempo do Colab; aumente se quiser mais precisão
!yolo task=detect mode=train model=yolo26n.pt data=Argoverse.yaml epochs=15 imgsz=640

O modelo treinado (pesos) fica salvo em **`runs/detect/trainX/weights/best.pt`**. Vamos carregar esses pesos para usar na detecção do vídeo.

##**Enviar o vídeo**

Faça upload do vídeo que você quer analisar.


In [ ]:
# Upload do vídeo direto do seu computador
from google.colab import files

uploaded = files.upload()
video_path = list(uploaded.keys())[0]
print(f"Vídeo enviado: {video_path}")

##**Executar detecção de objetos no vídeo, frame a frame**

Vamos carregar o modelo treinado com Argoverse e rodar a predição em modo `stream`, o que permite processar o vídeo frame a frame e imprimir quais objetos foram detectados em cada um.

In [ ]:
from ultralytics import YOLO
import glob

# Pega automaticamente o best.pt do treino mais recente
pesos_treinados = sorted(glob.glob('runs/detect/train*/weights/best.pt'))[-1]
print(f"Usando pesos: {pesos_treinados}")

model = YOLO(pesos_treinados)

# stream=True processa o vídeo frame a frame sem carregar tudo na memória de uma vez
# save=True salva o vídeo anotado (com as caixas desenhadas) em runs/detect/predictX
resultados = model.predict(source=video_path, save=True, conf=0.5, stream=True)

for i, r in enumerate(resultados):
    classes_detectadas = [model.names[int(c)] for c in r.boxes.cls]
    if classes_detectadas:
        print(f"Frame {i}: {classes_detectadas}")
    else:
        print(f"Frame {i}: nenhum objeto detectado")

O vídeo anotado (com as caixas desenhadas em cada objeto detectado) será salvo no diretório **"runs/detect/predictX"**. Você pode baixá-lo pelo painel de arquivos do Colab (ícone de pasta à esquerda).

##**Extra: contar quantas vezes cada objeto apareceu no vídeo todo**

In [ ]:
from collections import Counter

contador = Counter()
resultados = model.predict(source=video_path, conf=0.5, stream=True)

for r in resultados:
    classes_detectadas = [model.names[int(c)] for c in r.boxes.cls]
    contador.update(classes_detectadas)

print("Resumo de objetos detectados no vídeo inteiro:")
for classe, qtd in contador.most_common():
    print(f"  {classe}: {qtd} vezes")

##**Experimente você mesmo**

Agora que você treinou o YOLO com o Argoverse e rodou a detecção em vídeo, faça alguns experimentos:

1. Teste um novo vídeo
- Escolha um vídeo diferente (de trânsito, rua, estacionamento).
- Execute novamente a detecção.
- Observe quais objetos foram identificados em cada frame.

2. Altere o nível de confiança (conf)
- Rode o mesmo vídeo com conf=0.25 e depois conf=0.7. Compare os resultados.
- O que acontece quando aumentamos ou diminuímos esse valor?

3. Altere o número de épocas de treino (epochs)
- Treine novamente com mais épocas e compare a qualidade das detecções.

4. Analise os resultados
- Identifique pelo menos:
  - ✅ um objeto detectado corretamente;
  - ❌ uma detecção incorreta, se houver;
  - 🔍 um objeto presente no vídeo que não foi detectado, se houver.

Se você chegou até aqui, parabéns! 🎆 🔥

Fim!